# Evologics USBL — Position Visualisation

Plots ship track and modem-reported target positions on an interactive map.
Also derives absolute target positions from the raw USBL-frame XYZ offsets via
ship extrinsics (`scipy.spatial.transform.RigidTransform`) and ship attitude,
then overlays both on the same map for comparison.

USBL frame convention: X = right (starboard), Y = forward, Z = up.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import pymap3d
from plotly.subplots import make_subplots
from scipy.spatial.transform import RigidTransform, Rotation

In [ ]:
DEPLOYMENT: str = "qdc5ghs3_20210315_230947"

# NOTE: Deployments with Evologics USBL
# qd61g27j_20170523_040815     → 201705_silver_streak
# qdc5ghs3_20210315_230947     → 202102_poppag
# qdch0ftq_20170526_025746     → 201705_silver_streak
# qdch0ftq_20210315_034028     → 202102_poppag
# qdchdmy1_20170525_234624     → 201705_silver_streak
# qdchdmy1_20210315_081519     → 202102_poppag
# qtqxshxs_20150327_015552     → 201502_tasmania_bluefin
# qtqxshxs_20150328_000850     → 201502_tasmania_bluefin
# qtqxshxs_20150328_042551     → 201502_tasmania_bluefin

USBL_DIR: Path = Path("/data/exos_01/acfr_messages_v2_parsed")
USBL_FILE: Path = USBL_DIR / f"{DEPLOYMENT}_usbl_evologics.csv"

# Deployment label → ship sensor config key.
DEPLOYMENT_SHIP_CONFIG: dict[str, str] = {
    "qd61g27j_20170523_040815": "201705_silver_streak",
    "qdc5ghs3_20210315_230947": "202102_poppag",
    "qdch0ftq_20170526_025746": "201705_silver_streak",
    "qdch0ftq_20210315_034028": "202102_poppag",
    "qdchdmy1_20170525_234624": "201705_silver_streak",
    "qdchdmy1_20210315_081519": "202102_poppag",
    "qtqxshxs_20150327_015552": "201502_tasmania_bluefin",
    "qtqxshxs_20150328_000850": "201502_tasmania_bluefin",
    "qtqxshxs_20150328_042551": "201502_tasmania_bluefin",
}

# Ship extrinsics sourced from config/deployment_configs_all.toml.
# Translation (x, y, z) in metres — vessel frame: X=fwd, Y=stbd, Z=down.
# Rotation (phi, theta, psi) in radians — ZYX intrinsic Euler (yaw, pitch, roll).
SHIP_EXTRINSICS: dict[str, dict[str, float]] = {
    "201502_tasmania_bluefin": {
        "x": -11.06197886,
        "y": -6.65624488,
        "z": 9.99815286,
        "phi": 0.01590778,
        "theta": 0.01953048,
        "psi": -0.04636926,
    },
    "201705_silver_streak": {
        "x": -4.79286559,
        "y": 4.30177987,
        "z": 4.04273479,
        "phi": -0.00667731,
        "theta": 0.00317232,
        "psi": 0.47133307,
    },
    "202102_poppag": {
        "x": -9.4,
        "y": -3.72,
        "z": 7.49,
        "phi": 0.0,
        "theta": 0.23,
        "psi": -0.21,
    },
}

# Evologics USBL frame (X=right, Y=fwd, Z=up) → vessel body (X=fwd, Y=stbd, Z=down).
USBL_TO_VESSEL: np.ndarray = np.array(
    [[0.0, 1.0, 0.0], [1.0, 0.0, 0.0], [0.0, 0.0, -1.0]],
    dtype=np.float64,
)

## Load data

In [ ]:
REQUIRED_COLS: list[str] = [
    "timestamp",
    "target_latitude",
    "target_longitude",
    "target_depth",
    "target_x",
    "target_y",
    "target_z",
    "accuracy",
    "ship_latitude",
    "ship_longitude",
    "ship_roll",
    "ship_pitch",
    "ship_heading",
]

usbl: pd.DataFrame = pd.read_csv(
    USBL_FILE, parse_dates=["timestamp"], date_format="ISO8601"
)

missing: list[str] = [col for col in REQUIRED_COLS if col not in usbl.columns]
if missing:
    raise ValueError(f"CSV is missing required columns: {missing}")

print(f"Rows: {len(usbl)}")
usbl.head(3)

## Overview map: ship track and modem target positions

In [ ]:
t_s: np.ndarray = (usbl["timestamp"].astype(np.int64) / 1e9).to_numpy()
elapsed: np.ndarray = (t_s - t_s.min()) / 60.0

center_lat: float = float(usbl["target_latitude"].mean())
center_lon: float = float(usbl["target_longitude"].mean())

colorscale: str = "Viridis"

fig = go.Figure()

fig.add_trace(
    go.Scattermapbox(
        lat=usbl["ship_latitude"],
        lon=usbl["ship_longitude"],
        mode="lines+markers",
        marker=dict(
            size=4,
            color=elapsed,
            colorscale=colorscale,
            cmin=0,
            cmax=float(elapsed.max()),
            showscale=True,
            colorbar=dict(title="Elapsed (min)", x=0.92),
        ),
        line=dict(width=1, color="royalblue"),
        name="Ship track",
        hovertemplate=(
            "<b>Ship</b><br>"
            "Lat: %{lat:.6f}<br>"
            "Lon: %{lon:.6f}<br>"
            "<extra></extra>"
        ),
    )
)

fig.add_trace(
    go.Scattermapbox(
        lat=usbl["target_latitude"],
        lon=usbl["target_longitude"],
        mode="markers",
        marker=dict(
            size=6,
            color=elapsed,
            colorscale=colorscale,
            cmin=0,
            cmax=float(elapsed.max()),
            showscale=False,
        ),
        name="USBL target (modem)",
        customdata=np.stack(
            [usbl["target_depth"], usbl["accuracy"]],
            axis=1,
        ),
        hovertemplate=(
            "<b>USBL target (modem)</b><br>"
            "Lat: %{lat:.6f}<br>"
            "Lon: %{lon:.6f}<br>"
            "Depth: %{customdata[0]:.1f} m<br>"
            "Accuracy: %{customdata[1]:.2f} m<br>"
            "<extra></extra>"
        ),
    )
)

fig.update_layout(
    title=f"Ship track and USBL target positions — {DEPLOYMENT}",
    mapbox=dict(
        style="open-street-map",
        center=dict(lat=center_lat, lon=center_lon),
        zoom=14,
    ),
    legend=dict(x=0.01, y=0.99),
    height=700,
    margin=dict(l=0, r=0, t=40, b=0),
)

fig.show()

## Derive absolute target positions from XYZ offsets

The Evologics modem reports `target_x/y/z` in the USBL sensor frame
(X=right, Y=forward, Z=up). The conversion to absolute lat/lon follows four
steps:

1. **Frame flip** — map USBL axes to vessel body axes (X=fwd, Y=stbd, Z=down)
   via the constant flip matrix `USBL_TO_VESSEL`.
2. **Extrinsics** — apply the transceiver `RigidTransform` (rotation then
   translation) to express the target as an offset from the ship reference
   point in the vessel body frame.
3. **Ship attitude** — rotate from vessel body to NED using the per-ping
   heading / pitch / roll (ZYX intrinsic Euler).
4. **Geodetic conversion** — call `pymap3d.ned2geodetic` from the transceiver
   absolute position, which is derived separately from the extrinsics
   translation and the ship GPS position.

In [ ]:
ship_config_key: str = DEPLOYMENT_SHIP_CONFIG[DEPLOYMENT]
ext: dict[str, float] = SHIP_EXTRINSICS[ship_config_key]

# Build extrinsics RigidTransform: sensor frame → vessel body frame.
extrinsics_rotation: Rotation = Rotation.from_euler(
    "zyx", [ext["psi"], ext["theta"], ext["phi"]], degrees=False
)
extrinsics_translation: np.ndarray = np.array(
    [ext["x"], ext["y"], ext["z"]], dtype=np.float64
)
extrinsics_transform: RigidTransform = RigidTransform.from_components(
    translation=extrinsics_translation,
    rotation=extrinsics_rotation,
)

# Per-ping ship body → NED rotation (ZYX intrinsic: heading, pitch, roll).
ship_attitudes_ypr: np.ndarray = np.column_stack(
    [
        usbl["ship_heading"].to_numpy(),
        usbl["ship_pitch"].to_numpy(),
        usbl["ship_roll"].to_numpy(),
    ]
)
R_ship: Rotation = Rotation.from_euler("zyx", ship_attitudes_ypr, degrees=True)

# Step 1 — flip USBL frame → vessel body frame.
target_xyz_usbl: np.ndarray = np.column_stack(
    [
        usbl["target_x"].to_numpy(),
        usbl["target_y"].to_numpy(),
        usbl["target_z"].to_numpy(),
    ]
)
target_xyz_sensor: np.ndarray = (USBL_TO_VESSEL @ target_xyz_usbl.T).T

# Step 2 — extrinsics: sensor frame → vessel body frame (rotation only for
# NED; full transform gives vessel-frame offset for reference).
target_xyz_vessel: np.ndarray = extrinsics_transform.apply(target_xyz_sensor)

# Step 3 — ship attitude: vessel body → NED (target offset from transceiver).
target_body: np.ndarray = extrinsics_rotation.apply(target_xyz_sensor)
target_ned: np.ndarray = R_ship.apply(target_body)

# Step 4a — transceiver absolute position via extrinsics translation.
transceiver_ned: np.ndarray = R_ship.apply(
    np.tile(extrinsics_translation, (len(usbl), 1))
)
ship_lat: np.ndarray = usbl["ship_latitude"].to_numpy()
ship_lon: np.ndarray = usbl["ship_longitude"].to_numpy()
ship_alt: np.ndarray = np.zeros(len(usbl), dtype=np.float64)

transceiver_lat: np.ndarray
transceiver_lon: np.ndarray
transceiver_alt: np.ndarray
transceiver_lat, transceiver_lon, transceiver_alt = pymap3d.ned2geodetic(
    transceiver_ned[:, 0],
    transceiver_ned[:, 1],
    transceiver_ned[:, 2],
    ship_lat,
    ship_lon,
    ship_alt,
)

# Step 4b — target absolute position from transceiver.
derived_lat: np.ndarray
derived_lon: np.ndarray
derived_lat, derived_lon, _ = pymap3d.ned2geodetic(
    target_ned[:, 0],
    target_ned[:, 1],
    target_ned[:, 2],
    transceiver_lat,
    transceiver_lon,
    transceiver_alt,
)

print(f"Ship config : {ship_config_key}")
print(f"Derived rows: {len(derived_lat)}")
print(f"Lat range   : [{derived_lat.min():.6f}, {derived_lat.max():.6f}]")
print(f"Lon range   : [{derived_lon.min():.6f}, {derived_lon.max():.6f}]")

## Comparison map: modem vs. XYZ-derived target positions

In [ ]:
center_lat2: float = float(usbl["target_latitude"].mean())
center_lon2: float = float(usbl["target_longitude"].mean())

fig_cmp = go.Figure()

fig_cmp.add_trace(
    go.Scattermapbox(
        lat=usbl["ship_latitude"],
        lon=usbl["ship_longitude"],
        mode="lines+markers",
        marker=dict(
            size=4,
            color=elapsed,
            colorscale=colorscale,
            cmin=0,
            cmax=float(elapsed.max()),
            showscale=True,
            colorbar=dict(title="Elapsed (min)", x=0.92),
        ),
        line=dict(width=1, color="royalblue"),
        name="Ship track",
        hovertemplate=(
            "<b>Ship</b><br>"
            "Lat: %{lat:.6f}<br>"
            "Lon: %{lon:.6f}<br>"
            "<extra></extra>"
        ),
    )
)

fig_cmp.add_trace(
    go.Scattermapbox(
        lat=usbl["target_latitude"],
        lon=usbl["target_longitude"],
        mode="markers",
        marker=dict(
            size=7,
            color=elapsed,
            colorscale=colorscale,
            cmin=0,
            cmax=float(elapsed.max()),
            showscale=False,
        ),
        name="USBL target (modem)",
        customdata=np.stack(
            [usbl["target_depth"], usbl["accuracy"]],
            axis=1,
        ),
        hovertemplate=(
            "<b>USBL target (modem)</b><br>"
            "Lat: %{lat:.6f}<br>"
            "Lon: %{lon:.6f}<br>"
            "Depth: %{customdata[0]:.1f} m<br>"
            "Accuracy: %{customdata[1]:.2f} m<br>"
            "<extra></extra>"
        ),
    )
)

fig_cmp.add_trace(
    go.Scattermapbox(
        lat=derived_lat,
        lon=derived_lon,
        mode="markers",
        marker=dict(
            size=5,
            color="crimson",
            opacity=0.7,
        ),
        name="USBL target (XYZ-derived)",
        customdata=np.stack(
            [
                target_xyz_vessel[:, 0],
                target_xyz_vessel[:, 1],
                target_xyz_vessel[:, 2],
            ],
            axis=1,
        ),
        hovertemplate=(
            "<b>USBL target (XYZ-derived)</b><br>"
            "Lat: %{lat:.6f}<br>"
            "Lon: %{lon:.6f}<br>"
            "Vessel X: %{customdata[0]:.2f} m<br>"
            "Vessel Y: %{customdata[1]:.2f} m<br>"
            "Vessel Z: %{customdata[2]:.2f} m<br>"
            "<extra></extra>"
        ),
    )
)

fig_cmp.update_layout(
    title=f"Modem vs. XYZ-derived USBL target positions — {DEPLOYMENT}",
    mapbox=dict(
        style="open-street-map",
        center=dict(lat=center_lat2, lon=center_lon2),
        zoom=14,
    ),
    legend=dict(x=0.01, y=0.99),
    height=700,
    margin=dict(l=0, r=0, t=40, b=0),
)

fig_cmp.show()

## Time series: modem vs. XYZ-derived target position (NED, metres)

In [ ]:
# Reference origin: mean ship position over the deployment.
ref_lat: float = float(usbl["ship_latitude"].mean())
ref_lon: float = float(usbl["ship_longitude"].mean())
ref_alt: float = 0.0


def to_ned(lat: np.ndarray, lon: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    north, east, _ = pymap3d.geodetic2ned(
        lat,
        lon,
        np.zeros_like(lat),
        ref_lat,
        ref_lon,
        ref_alt,
    )
    return north, east


modem_n, modem_e = to_ned(
    usbl["target_latitude"].to_numpy(), usbl["target_longitude"].to_numpy()
)
derived_n, derived_e = to_ned(derived_lat, derived_lon)
ship_n, ship_e = to_ned(
    usbl["ship_latitude"].to_numpy(), usbl["ship_longitude"].to_numpy()
)

horizontal_error: np.ndarray = np.sqrt(
    (modem_n - derived_n) ** 2 + (modem_e - derived_e) ** 2
)

fig_ned = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    subplot_titles=(
        "North offset (m)",
        "East offset (m)",
        "Horizontal error: modem vs. XYZ-derived (m)",
    ),
    vertical_spacing=0.07,
)

for row, (
    modem_vals,
    derived_vals,
    ship_vals,
    modem_name,
    derived_name,
    ship_name,
) in enumerate(
    [
        (
            modem_n,
            derived_n,
            ship_n,
            "Modem (N)",
            "XYZ-derived (N)",
            "Ship (N)",
        ),
        (
            modem_e,
            derived_e,
            ship_e,
            "Modem (E)",
            "XYZ-derived (E)",
            "Ship (E)",
        ),
    ],
    start=1,
):
    fig_ned.add_trace(
        go.Scatter(
            x=usbl["timestamp"],
            y=modem_vals,
            mode="lines+markers",
            marker=dict(size=4),
            name=modem_name,
            line=dict(color="steelblue" if row == 1 else "darkorange"),
        ),
        row=row,
        col=1,
    )
    fig_ned.add_trace(
        go.Scatter(
            x=usbl["timestamp"],
            y=derived_vals,
            mode="lines+markers",
            marker=dict(size=4),
            name=derived_name,
            line=dict(color="crimson" if row == 1 else "purple", dash="dash"),
        ),
        row=row,
        col=1,
    )
    fig_ned.add_trace(
        go.Scatter(
            x=usbl["timestamp"],
            y=ship_vals,
            mode="lines",
            name=ship_name,
            line=dict(color="grey", dash="dot", width=1),
        ),
        row=row,
        col=1,
    )

fig_ned.add_trace(
    go.Scatter(
        x=usbl["timestamp"],
        y=horizontal_error,
        mode="lines+markers",
        marker=dict(size=4),
        name="Horizontal error",
        line=dict(color="seagreen"),
    ),
    row=3,
    col=1,
)

fig_ned.update_yaxes(title_text="m", row=1, col=1)
fig_ned.update_yaxes(title_text="m", row=2, col=1)
fig_ned.update_yaxes(title_text="m", rangemode="tozero", row=3, col=1)

fig_ned.update_layout(
    title=f"Target position NED — {DEPLOYMENT} (ref: mean ship position)",
    height=750,
    legend=dict(x=0.01, y=0.99),
)

fig_ned.show()

## Time series: target XYZ offsets and depth

In [ ]:
fig_xyz = make_subplots(
    rows=4,
    cols=1,
    shared_xaxes=True,
    subplot_titles=(
        "Target X — USBL frame right (m)",
        "Target Y — USBL frame forward (m)",
        "Target Z — USBL frame up (m)",
        "Target depth (m)",
    ),
    vertical_spacing=0.06,
)

for row, (col, color) in enumerate(
    [
        ("target_x", "steelblue"),
        ("target_y", "darkorange"),
        ("target_z", "seagreen"),
        ("target_depth", "crimson"),
    ],
    start=1,
):
    fig_xyz.add_trace(
        go.Scatter(
            x=usbl["timestamp"],
            y=usbl[col],
            mode="lines+markers",
            marker=dict(size=4),
            name=col,
            line=dict(color=color),
        ),
        row=row,
        col=1,
    )

fig_xyz.update_yaxes(autorange="reversed", row=4, col=1)

fig_xyz.update_layout(
    title=f"Target XYZ offsets (USBL frame) and depth — {DEPLOYMENT}",
    height=900,
    showlegend=False,
)

fig_xyz.show()